# Chapter 2 — Model I/O: Prompt Templates & Output Parsing

Hands-On LangChain. Code targets **LangChain 1.x** (Ollama, LCEL, Pydantic structured output).

This notebook accompanies Chapter 2. Run it top to bottom; outputs are captured under each cell and mirrored into the chapter text.

**Requirements**
```bash
pip install --upgrade langchain langchain-ollama langchain-community pydantic ollama -q
```
Configure `MODEL` in `.env` and start Ollama with the selected model pulled; the notebook loads the shared Ollama configuration.

## 2.0 Setup

In [ ]:
import os
import ollama
from dotenv import load_dotenv, find_dotenv
from model_provider import MODEL, get_chat_model

load_dotenv(find_dotenv(usecwd=True))
llm_model = MODEL
client = ollama.Client(host=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"))
chat = get_chat_model(temperature=0.0)
print("Using model:", llm_model)

### A quick "before LangChain" baseline

Before we use LangChain, here is the raw way to call the model. It works, but notice how
much glue code you write per call — building the message list, picking apart the response.

In [ ]:
def get_completion(prompt, model=llm_model, temperature=0.0, max_completion_tokens=500):
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature, "num_predict": max_completion_tokens},
    )
    return response.message.content

## 2.1 Prompt Templates with LangChain

Say a customer writes in an informal "pirate" English and we want it rephrased into calm,
respectful American English.

In [ ]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
style = """American English in a calm and respectful tone"""

prompt = f"""Translate the text that is delimited by triple backticks
into a style that is {style}.
text: ```{customer_email}```
"""
print(prompt)

In [ ]:
response = get_completion(prompt)
response

That works, but if you have customers writing in many languages and tones, you end up
generating a whole sequence of ad-hoc prompts. LangChain's **prompt templates** let you write
the prompt once and reuse it. Let's switch to LangChain's chat model.

In [ ]:
# `chat` was created from the shared Ollama provider configuration above.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

prompt_template = ChatPromptTemplate.from_template(template_string)

In [ ]:
# The template knows its input variables
prompt_template.input_variables

In [ ]:
customer_style = """American English in a calm and respectful tone"""

customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email,
)

print(type(customer_messages))
print(type(customer_messages[0]))

Now pass the formatted messages to the model. In LangChain 1.x you call `.invoke()` (the old callable `chat(messages)` syntax is removed).

In [ ]:
customer_response = chat.invoke(customer_messages)
print(customer_response.content)

Because the prompt is a reusable template, we can swap in a completely different style and text without rewriting anything — e.g. translating a support agent's blunt reply into polite English pirate.

In [ ]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """a polite tone that speaks in English Pirate"""

service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply,
)
print(service_messages[0].content)

In [ ]:
service_response = chat.invoke(service_messages)
print(service_response.content)

### The modern shortcut: LCEL

In LangChain 1.x you usually wire the prompt and model together with the **pipe** (`|`) operator
into a single runnable, then `.invoke()` it with the variables. This is the idiom you'll see for
the rest of the book.

In [ ]:
chain = prompt_template | chat
result = chain.invoke({"style": customer_style, "text": customer_email})
print(result.content)

## 2.2 Output Parsing

When you build real applications you often want **structured** output (e.g. JSON) with specific
keys, not a free-form string. Let's extract three fields from a product review:
`gift`, `delivery_days`, and `price_value`.

In [ ]:
customer_review = """\
This leaf blower is pretty amazing. It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

First, the naive approach: ask for JSON in the prompt and see what comes back.

In [ ]:
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price, \
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

prompt_template = ChatPromptTemplate.from_template(review_template)
messages = prompt_template.format_messages(text=customer_review)
response = chat.invoke(messages)
print(response.content)

It *looks* like JSON, but it is just a string — so you can't index into it like a dict.

In [ ]:
type(response.content)

In [ ]:
# This raises AttributeError, because a str has no .get()
try:
    response.content.get("gift")
except AttributeError as e:
    print("AttributeError:", e)

### The 1.x way: `with_structured_output` + Pydantic

The old `ResponseSchema` / `StructuredOutputParser` API has been removed. The modern, type-safe
replacement is to define a **Pydantic model** and call `chat.with_structured_output(...)`. The
model is guaranteed to return an instance of your schema — no manual parsing, no string juggling.

In [ ]:
from pydantic import BaseModel, Field

class ReviewInfo(BaseModel):
    """Information extracted from a product review."""
    gift: bool = Field(description="Was the item purchased as a gift for someone else? "
                                   "True if yes, False if not or unknown.")
    delivery_days: int = Field(description="How many days did it take to arrive? -1 if unknown.")
    price_value: list[str] = Field(description="Sentences about the value or price.")

In [ ]:
structured_chat = chat.with_structured_output(ReviewInfo)

extract_prompt = ChatPromptTemplate.from_template(
    "Extract the requested fields from this product review:\n\n{text}"
)
chain = extract_prompt | structured_chat

info = chain.invoke({"text": customer_review})
info

`info` is a real `ReviewInfo` object with typed attributes — `gift` is a bool, `delivery_days` is an int, `price_value` is a list.

In [ ]:
print("gift           :", info.gift, type(info.gift).__name__)
print("delivery_days  :", info.delivery_days, type(info.delivery_days).__name__)
print("price_value    :", info.price_value)

### Alternative: `JsonOutputParser` (dict instead of object)

If you prefer a plain `dict` and want to inject format instructions into the prompt yourself,
`JsonOutputParser` is the LCEL-friendly equivalent of the old parser.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser(pydantic_object=ReviewInfo)

prompt = ChatPromptTemplate.from_template(
    "Extract the fields from the review.\n{format_instructions}\n\nreview: {text}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | chat | parser
output_dict = chain.invoke({"text": customer_review})
print(type(output_dict))
output_dict

In [ ]:
# Now it really is a dict — you can index into it
output_dict["gift"]